# 🤖 PR Governance Agent - Colab Edition
This notebook allows you to run and test the PR Governance Agent Brains (Llama 3, Gemini, or Bedrock) directly inside Google Colab.

## 1. Setup Environment & Clone Repository
First, we install the necessary dependencies and clone the required repository.

In [ ]:
!pip install requests groq google-generativeai pydantic boto3

import os

# Clone the repository
REPO_URL = input("Repository URL: ")
!git clone {REPO_URL}

# Extract repo name for directory change
repo_name = REPO_URL.split('/')[-1].replace('.git', '')
%cd {repo_name}

print("\n✅ Environment Setup Complete")

## 2. Configure API Keys
Securely enter your API keys. Only provide the keys for the models you intend to test.

In [ ]:
from getpass import getpass
import os

print("Enter your GitHub PAT (Required for fetching PR info):")
os.environ["GITHUB_TOKEN"] = getpass("GitHub PAT: ")

print("\nEnter your Groq API Key (For Llama 3 Brain):")
os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")

print("\nEnter your Google Gemini API Key (For Gemini Brain):")
os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key: ")

print("\n✅ Keys Configured")

## 3. Configure AWS Credentials (Optional, for Bedrock)
Only needed if you want to test the AWS Bedrock Managed Agent.

In [ ]:
print("Enter AWS Access Key ID (Leave blank to skip):")
aws_key = getpass("AWS Access Key: ")
if aws_key:
    os.environ["AWS_ACCESS_KEY_ID"] = aws_key
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS Secret Key: ")
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
    print("✅ AWS Configured")
else:
    print("▶️ Skipped AWS Configuration (Bedrock agent will not work)")

## 4. Test Local Heartbeat / Brain
Enter the Repo ID and PR ID you want to analyze.

In [ ]:
TEST_REPO = input("Test Repo (owner/repo): ")
TEST_PR = input("Test PR Number: ")

## 5. Test the Llama 3 Brain (Groq)

In [ ]:
import sys
import json

if "PR-Agent-Brain-Simple" not in sys.path:
    sys.path.insert(0, "PR-Agent-Brain-Simple")

try:
    if 'lambda_function' in sys.modules:
        del sys.modules['lambda_function']
    import lambda_function as llama_brain
    
    payload = {
        "installation": {"id": 12345678},
        "repo_full_name": TEST_REPO,
        "pr_number": TEST_PR,
        "is_pull_request": True,
        "user_message": "Analyze this PR",
        "sender_name": "ColabUser"
    }
    
    print("⏳ Running Llama 3 Analysis...")
    response = llama_brain.lambda_handler(payload, None)
    print("\n✅ Llama 3 Agent Response:")
    print(json.dumps(response, indent=2))
except Exception as e:
    print(f"❌ Error: {e}")


## 6. Test the Gemini Brain

In [ ]:
import sys
import json

if "PR-Agent-Brain-Simple" in sys.path:
    sys.path.remove("PR-Agent-Brain-Simple")
    
if "PR-Agent-Brain-Gemini" not in sys.path:
    sys.path.insert(0, "PR-Agent-Brain-Gemini")

try:
    if 'lambda_function' in sys.modules:
        del sys.modules['lambda_function']
    import lambda_function as gemini_brain
    
    payload = {
        "installation": {"id": 12345678},
        "repo_full_name": TEST_REPO,
        "pr_number": TEST_PR,
        "is_pull_request": True,
        "user_message": "Analyze this PR deeply",
        "sender_name": "ColabUser"
    }
    
    print("⏳ Running Gemini Analysis...")
    response = gemini_brain.lambda_handler(payload, None)
    print("\n✅ Gemini Agent Response:")
    print(json.dumps(response, indent=2))
except Exception as e:
    print(f"❌ Error: {e}")
